# Capitolo 9: Sintesi dei Risultati e Benchmark Comparativo

## Obiettivo
Riassumere l'intero percorso di ricerca FlowStitch, presentare i risultati 
quantitativi e documentare le lezioni apprese.


## 9.1 Il Percorso di Ricerca

| Capitolo | Domanda | Risposta | Metodo Chiave |
|----------|---------|----------|---------------|
| 1 | Come è strutturato lo spazio latente? | Energia e attenzione si allineano a $t=0$ | KPE, Cross-Attention |
| 2 | Quale variabile manipolare? | $v_0$ (velocità), non $x_{\text{pred}}$ | Conservazione isotropia |
| 3 | Le hard mask funzionano? | **No** — violano Picard-Lindelöf | Analisi Lipschitz |
| 4 | Il blending continuo risolve? | Parzialmente — appare il Ghosting | Alpha matte continuo |
| 5 | Come eliminare il ghosting? | 6 decoder iterativi fino al vincitore | SvdSeeded+GraphDiffusion |
| 6 | Esiste un metodo senza soglie? | **Sì** — Vettore di Fiedler | Spectral Matting, TDA |
| 7 | Come gestire multi-oggetto? | Modulazione semantica + sigmoid | DNA Vettoriale |
| 8 | La pipeline è coerente? | Sì — tutti i metodi integrati | FlowStitch package |


## 9.2 Esperimenti Falliti e Lezioni Apprese

| Esperimento | Capitolo | Perché è fallito | Cosa abbiamo imparato |
|-------------|----------|------------------|----------------------|
| Hard masking | 3 | Discontinuità $L \to \infty$ al bordo | Picard-Lindelöf richiede continuità Lipschitz |
| Chebyshev gating puro | 5 | Vuoto Termodinamico (maschera a ciambella) | L'energia è bassa al centro degli oggetti piatti |
| Seed da media heads | 5 | Rumore semantico nel GraphDiffusion | La media è subottimale — SVD estrae il consenso |
| ARPACK a risoluzione piena | 6 | Gap spettrale $\Delta\lambda \approx 10^{-6}$ | Matrici $4096 \times 4096$ mal condizionate |
| Co-Embedding ibrido | 6 | ArpackNoConvergence dopo 40961 iter | Necessaria decimazione bilineare a $32 \times 32$ |


## 9.3 Benchmark Quantitativo dei Metodi di Estrazione

In [ ]:
import os, sys, torch
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, os.path.abspath('..'))

from flowstitch.core.tokenizer_utils import find_token_indices
from flowstitch.extraction import (
    extract_attention_mask, otsu_threshold,
    compute_fiedler_mask, compute_chebyshev_threshold,
    hybrid_semantic_decomposition, extract_tda_mask
)
from flowstitch.evaluation.metrics import dice_coefficient, iou_score

# Benchmark su campioni singoli
samples = [
    ("a_red_cube", "cube"),
    ("a_blue_sphere", "sphere"),
]

results = []
for folder, target in samples:
    db_path = f"../data/dataset_v1/{folder}"
    if not os.path.exists(db_path):
        print(f"Skip: {db_path}")
        continue
    
    attn = torch.load(os.path.join(db_path, "attention_maps.pt"), map_location="cpu", weights_only=True)
    v0 = torch.load(os.path.join(db_path, "v0_velocity.pt"), map_location="cpu", weights_only=True)
    
    try:
        from transformers import T5Tokenizer
        tok = T5Tokenizer.from_pretrained("google/t5-v1_1-xxl", legacy=True)
        tidx = find_token_indices(tok, f"a {target}", target)
    except:
        tidx = [2]
    
    attn_mask = extract_attention_mask(attn, tidx)
    energy = torch.norm(v0, p=2, dim=-1, keepdim=True)
    
    # Reference: TDA
    ref = extract_tda_mask(v0, attn_mask).float()
    
    methods = {
        "Otsu": otsu_threshold(attn_mask).float(),
        "Chebyshev": compute_chebyshev_threshold(energy, k=1.0).float(),
        "Fiedler": compute_fiedler_mask(v0, target_resolution=32).float(),
        "Hybrid": hybrid_semantic_decomposition(v0, attn_mask, target_resolution=32).float(),
    }
    
    for method_name, mask in methods.items():
        dice = dice_coefficient(mask, ref)
        iou_val = iou_score(mask, ref)
        results.append({"sample": folder, "method": method_name, "DICE": dice, "IoU": iou_val})
        
    print(f"Processato: {folder}")

# Tabella
print("\n=== Risultati per Metodo (media) ===")
from collections import defaultdict
agg = defaultdict(lambda: {"DICE": [], "IoU": []})
for r in results:
    agg[r["method"]]["DICE"].append(r["DICE"])
    agg[r["method"]]["IoU"].append(r["IoU"])

for method, vals in agg.items():
    d = sum(vals["DICE"])/len(vals["DICE"])
    i = sum(vals["IoU"])/len(vals["IoU"])
    print(f"  {method:15s} | DICE={d:.4f} | IoU={i:.4f}")


## 9.4 Conclusioni

### Contributi Principali
1. **Conservazione della varianza**: dimostrazione che il blending $\sqrt{\cdot}$ 
   è necessario per mantenere $\sigma^2 = 1.0$ (Cap. 4)
2. **Vuoto Termodinamico**: scoperta e formalizzazione della dualità energia-densità (Cap. 5)
3. **6 decoder iterativi**: percorso esplorativo documentato dal gating statistico alla 
   diffusione spettrale (Cap. 5)
4. **Vettore di Fiedler**: segmentazione threshold-free che risolve il Vuoto (Cap. 6)
5. **DNA Vettoriale**: decomposizione multi-oggetto con validazione ortogonalità (Cap. 7)
6. **ARPACK failure → decimazione**: soluzione pratica al mal condizionamento (Cap. 6)

### Limiti
- Validazione solo su prompt sintetici (geometrie semplici)
- Nessun confronto con ground truth umano
- Pipeline non testata su FLUX.1-dev (solo schnell)

### Codice
Tutto il codice è integrato nel pacchetto `flowstitch/`:
- `extraction/`: 6 metodi di estrazione maschera + 5 decoder sperimentali + decomposizione multi-oggetto
- `stitching/`: KTS, EMA, SemanticGrafting, ODE perturbation
- `evaluation/`: DICE, IoU, CLIPScore, BenchmarkRunner
- `pipelines/`: dataset generation, mask compilation, latent stitching
